# 01 Data Collection and Cleaning

This notebook loads the raw job posting dataset, checks its structure, cleans inconsistent values, validates missing data, and saves a processed version for analysis.

Project: SkillMap — UAE–Canada Tech Job Market Intelligence Dashboard

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_DATA_PATH = Path("../data/raw/job_postings_sample.csv")
PROCESSED_DATA_PATH = Path("../data/processed/cleaned_jobs.csv")
SKILLS_OUTPUT_PATH = Path("../data/processed/job_skills.csv")

In [3]:
df = pd.read_csv(RAW_DATA_PATH)

df.head()

,job_id,collection_date,country,city,job_title,company,source,job_url,employment_type,work_mode,seniority,salary_min,salary_max,currency,years_experience_min,years_experience_max,education_requirement,description_short,skills_text,notes
0,UAE_001,2026-05-20,UAE,Dubai,Product Data Analyst Intern,TikTok,GulfTalent,https://www.gulftalent.com/uae/jobs/product-da...,Full-time,NaN,Internship,NaN,NaN,NaN,NaN,NaN,NaN,Internship analyzing product performance trend...,"SQL, Excel, Tableau, Data Visualization, Dashb...",Internship requires at least 3-month commitment.
1,UAE_002,2026-05-20,UAE,Dubai,Project Data Analyst Intern,CAGS Management Services DMCC,Bayt,https://www.bayt.com/en/uae/jobs/project-data-...,NaN,NaN,Internship,NaN,NaN,NaN,NaN,NaN,NaN,Internship supporting ERP implementation throu...,"Data Cleaning, Quality Assurance, Manual Testi...",Bayt notes the post was translated by AI.
2,UAE_003,2026-05-20,UAE,Dubai,Data Analyst and Visual Content Designer (UAE ...,Genius HRTech Services,Bayt,https://www.bayt.com/en/uae/jobs/data-analyst-...,Full-time,NaN,Entry-level,14815.0,18519.0,AED,1.0,3.0,Bachelor's degree / higher diploma,"Entry-level role analyzing HSE datasets, build...","Excel, Power BI, Dashboarding, Reporting, Data...",UAE nationals only.
3,UAE_004,2026-05-20,UAE,Dubai,Associate Data Analyst- UAE Nationals only,Delivery Hero SE,Bayt,https://www.bayt.com/en/uae/jobs/associate-dat...,NaN,NaN,Associate,NaN,NaN,NaN,2.0,3.0,"Bachelor's degree in Data Science, Statistics,...",Early-career analytics role supporting product...,"SQL, Excel, Tableau, Power BI, Looker, Python,...",UAE nationals only; early-career wording but r...
4,UAE_005,2026-05-20,UAE,Dubai,Associate Data Analyst,Bayut | dubizzle,GulfTalent,https://www.gulftalent.com/uae/jobs/associate-...,Full-time,NaN,Associate,NaN,NaN,NaN,2.0,3.0,NaN,"Associate strategy analytics role using SQL, E...","SQL, Excel, Dashboarding, Reporting, Data Visu...",NaN


In [4]:
df.shape

(20, 20)

In [5]:
df.columns.tolist()

['job_id',
 'collection_date',
 'country',
 'city',
 'job_title',
 'company',
 'source',
 'job_url',
 'employment_type',
 'work_mode',
 'seniority',
 'salary_min',
 'salary_max',
 'currency',
 'years_experience_min',
 'years_experience_max',
 'education_requirement',
 'description_short',
 'skills_text',
 'notes']

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   job_id                 20 non-null     str    
 1   collection_date        20 non-null     str    
 2   country                20 non-null     str    
 3   city                   20 non-null     str    
 4   job_title              20 non-null     str    
 5   company                20 non-null     str    
 6   source                 20 non-null     str    
 7   job_url                20 non-null     str    
 8   employment_type        18 non-null     str    
 9   work_mode              11 non-null     str    
 10  seniority              20 non-null     str    
 11  salary_min             11 non-null     float64
 12  salary_max             11 non-null     float64
 13  currency               11 non-null     str    
 14  years_experience_min   13 non-null     float64
 15  years_experience_ma

In [7]:
missing_summary = df.isna().sum().sort_values(ascending=False)
missing_summary

salary_max               9
currency                 9
work_mode                9
salary_min               9
years_experience_max     7
years_experience_min     7
education_requirement    5
notes                    4
employment_type          2
country                  0
collection_date          0
job_id                   0
city                     0
seniority                0
source                   0
job_url                  0
job_title                0
company                  0
description_short        0
skills_text              0
dtype: int64

In [8]:
clean_df = df.copy()

In [9]:
text_columns = [
    "job_id",
    "country",
    "city",
    "job_title",
    "company",
    "source",
    "job_url",
    "employment_type",
    "work_mode",
    "seniority",
    "currency",
    "education_requirement",
    "description_short",
    "skills_text",
    "notes"
]

for col in text_columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

In [10]:
clean_df["collection_date"] = pd.to_datetime(clean_df["collection_date"], errors="coerce")

In [11]:
numeric_columns = [
    "salary_min",
    "salary_max",
    "years_experience_min",
    "years_experience_max"
]

for col in numeric_columns:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")

In [12]:
clean_df["country"] = clean_df["country"].replace({
    "UAE": "UAE",
    "United Arab Emirates": "UAE",
    "Canada": "Canada"
})

In [13]:
clean_df["country"].value_counts()

country
UAE       10
Canada    10
Name: count, dtype: Int64

In [14]:
clean_df["employment_type"] = clean_df["employment_type"].replace({
    "Full Time": "Full-time",
    "Full time": "Full-time",
    "Permanent employment Full time": "Full-time",
    "Full-time Internship": "Internship",
    "Full-time Internship/Co-op": "Internship/Co-op",
    "Full-time Internship / Co-op": "Internship/Co-op",
    "Internship / Co-op": "Internship/Co-op"
})

In [15]:
clean_df["employment_type"].value_counts(dropna=False)

employment_type
Full-time           10
Internship/Co-op     5
<NA>                 2
Internship           2
Contract             1
Name: count, dtype: Int64

In [16]:
clean_df["work_mode"] = clean_df["work_mode"].replace({
    "On site": "On-site",
    "Onsite": "On-site",
    "In person": "On-site",
    "In-person": "On-site",
    "Remote": "Remote",
    "Hybrid": "Hybrid"
})

In [17]:
clean_df["work_mode"].value_counts(dropna=False)

work_mode
<NA>       9
Hybrid     6
On-site    4
Remote     1
Name: count, dtype: Int64

In [18]:
clean_df["seniority"] = clean_df["seniority"].replace({
    "Intern": "Internship",
    "Entry level": "Entry-level",
    "Early-career": "Junior",
    "Coop": "Co-op",
    "Co-op": "Co-op",
    "Associate": "Associate",
    "Junior": "Junior",
    "Internship": "Internship"
})

In [19]:
clean_df["seniority"].value_counts(dropna=False)

seniority
Internship     8
Junior         5
Entry-level    3
Associate      2
Co-op          2
Name: count, dtype: Int64

In [20]:
def detect_salary_period(notes):
    if pd.isna(notes):
        return pd.NA
    
    notes_lower = str(notes).lower()
    
    if "hourly" in notes_lower:
        return "Hourly"
    if "annually" in notes_lower or "annual" in notes_lower:
        return "Annual"
    if "monthly" in notes_lower:
        return "Monthly"
    
    return pd.NA

clean_df["salary_period"] = clean_df["notes"].apply(detect_salary_period)

In [21]:
clean_df[["job_id", "salary_min", "salary_max", "currency", "salary_period", "notes"]]

,job_id,salary_min,salary_max,currency,salary_period,notes
0,UAE_001,NaN,NaN,<NA>,NaN,Internship requires at least 3-month commitment.
1,UAE_002,NaN,NaN,<NA>,NaN,Bayt notes the post was translated by AI.
2,UAE_003,14815.0,18519.0,AED,NaN,UAE nationals only.
3,UAE_004,NaN,NaN,<NA>,NaN,UAE nationals only; early-career wording but r...
4,UAE_005,NaN,NaN,<NA>,NaN,<NA>
5,UAE_006,NaN,NaN,<NA>,NaN,<NA>
6,UAE_007,NaN,NaN,<NA>,NaN,<NA>
7,UAE_008,NaN,NaN,<NA>,NaN,Remote role; posting emphasizes spreadsheets a...
8,UAE_009,NaN,NaN,<NA>,NaN,<NA>
9,UAE_010,NaN,NaN,<NA>,NaN,LinkedIn labels entry-level; posting asks for ...


In [22]:
clean_df["has_salary"] = clean_df["salary_min"].notna() | clean_df["salary_max"].notna()

In [23]:
def categorize_experience(min_years):
    if pd.isna(min_years):
        return "Not specified"
    if min_years == 0:
        return "0 years / student-friendly"
    if min_years <= 1:
        return "1 year"
    if min_years <= 2:
        return "2 years"
    if min_years <= 3:
        return "3 years"
    return "4+ years"

clean_df["experience_category"] = clean_df["years_experience_min"].apply(categorize_experience)

In [24]:
clean_df["experience_category"].value_counts()

experience_category
Not specified                 7
0 years / student-friendly    5
1 year                        4
2 years                       4
Name: count, dtype: int64

In [25]:
clean_df["entry_level_with_high_experience"] = (
    clean_df["seniority"].isin(["Entry-level", "Junior", "Associate"]) &
    (clean_df["years_experience_min"] >= 3)
)

In [26]:
clean_df[clean_df["entry_level_with_high_experience"] == True][
    ["job_id", "job_title", "company", "seniority", "years_experience_min", "years_experience_max", "notes"]
]

,job_id,job_title,company,seniority,years_experience_min,years_experience_max,notes


In [27]:
skills_df = (
    clean_df[["job_id", "country", "city", "job_title", "company", "skills_text"]]
    .dropna(subset=["skills_text"])
    .assign(skill=lambda x: x["skills_text"].str.split(","))
    .explode("skill")
)

skills_df["skill"] = skills_df["skill"].astype("string").str.strip()

skills_df = skills_df[skills_df["skill"].notna() & (skills_df["skill"] != "")]

skills_df.head(10)

,job_id,country,city,job_title,company,skills_text,skill
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",SQL
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Excel
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Tableau
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Data Visualization
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Dashboarding
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Reporting
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Stakeholder Communication
0,UAE_001,UAE,Dubai,Product Data Analyst Intern,TikTok,"SQL, Excel, Tableau, Data Visualization, Dashb...",Presentation Skills
1,UAE_002,UAE,Dubai,Project Data Analyst Intern,CAGS Management Services DMCC,"Data Cleaning, Quality Assurance, Manual Testi...",Data Cleaning
1,UAE_002,UAE,Dubai,Project Data Analyst Intern,CAGS Management Services DMCC,"Data Cleaning, Quality Assurance, Manual Testi...",Quality Assurance


In [28]:
top_skills = skills_df["skill"].value_counts()
top_skills

skill
Reporting                                                                                 16
Stakeholder Communication                                                                 16
SQL                                                                                       10
Technical Documentation                                                                   10
Data Visualization                                                                         9
Dashboarding                                                                               9
Excel                                                                                      8
Power BI                                                                                   8
Python                                                                                     8
Presentation Skills                                                                        7
Data Cleaning                                                   

In [29]:
skills_by_country = (
    skills_df
    .groupby(["country", "skill"])
    .size()
    .reset_index(name="count")
    .sort_values(["country", "count"], ascending=[True, False])
)

skills_by_country.head(20)

,country,skill,count
15,Canada,Reporting,8
17,Canada,Stakeholder Communication,8
20,Canada,Technical Documentation,8
12,Canada,Python,6
16,Canada,SQL,6
3,Canada,Data Visualization,4
10,Canada,Power BI,4
13,Canada,Quality Assurance,4
1,Canada,Dashboarding,3
2,Canada,Data Cleaning,3


In [30]:
clean_df.to_csv(PROCESSED_DATA_PATH, index=False)
skills_df.to_csv(SKILLS_OUTPUT_PATH, index=False)

print(f"Cleaned jobs saved to: {PROCESSED_DATA_PATH}")
print(f"Job skills saved to: {SKILLS_OUTPUT_PATH}")

Cleaned jobs saved to: ..\data\processed\cleaned_jobs.csv
Job skills saved to: ..\data\processed\job_skills.csv


In [31]:
test_cleaned = pd.read_csv(PROCESSED_DATA_PATH)
test_skills = pd.read_csv(SKILLS_OUTPUT_PATH)

print(test_cleaned.shape)
print(test_skills.shape)

(20, 24)
(133, 7)


## Cleaning Summary

The raw 20-row dataset was loaded and validated. Text fields were stripped of extra whitespace, dates and numeric fields were converted to appropriate types, employment type/work mode/seniority values were standardized, and derived columns were created for salary availability, salary period, experience category, and entry-level roles requiring higher experience.

Two processed files were saved:

- `data/processed/cleaned_jobs.csv`
- `data/processed/job_skills.csv`

These files will be used in the next phases for SQL analysis and dashboard development.